<a href="https://colab.research.google.com/github/mejia080902-bit/pymc-examples/blob/main/dos_servidores_en_serie.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import math
from collections import deque
import numpy as np


def simulate_tandem_mm1(lambda_rate, mu1, mu2, N=100_000, warmup=2_000, seed=123):
    """
    Simula una línea de espera con dos servidores EN SERIE:
      Llegadas -> Cola1/Serv1 -> Cola2/Serv2 -> Salida
    Interarribos ~ Exp(lambda_rate), servicios ~ Exp(mu1), Exp(mu2).

    Retorna promedios:
      W  = tiempo promedio en el sistema (cola1 + serv1 + cola2 + serv2)
      Wq = tiempo promedio formado en las filas (cola1 + cola2)

    warmup: número de clientes iniciales descartados para reducir transitorio.
    """
    if not (lambda_rate < mu1 and lambda_rate < mu2):
        raise ValueError("Sistema inestable: se requiere lambda < mu1 y lambda < mu2.")

    rng = np.random.default_rng(seed)

    # Tiempos por cliente (id = 0..N-1)
    arr1     = np.empty(N)  # llegada al sistema / a S1
    s1_start = np.empty(N)  # inicio de servicio en S1
    dep1     = np.empty(N)  # fin de servicio en S1 (=> llegada a S2)
    arr2     = np.empty(N)  # llegada a S2 (igual a dep1)
    s2_start = np.empty(N)  # inicio de servicio en S2
    dep2     = np.empty(N)  # salida del sistema

    # Estado del sistema
    q1, q2 = deque(), deque()
    server1_busy = False
    server2_busy = False
    current1 = None
    current2 = None

    # Reloj y próximos sucesos (LS = {tLL, t1, t2})
    t = 0.0
    n_arr = 0
    n_dep = 0

    tLL = rng.exponential(1.0 / lambda_rate)  # primera llegada
    t1 = math.inf  # próxima salida de S1
    t2 = math.inf  # próxima salida de S2

    while n_dep < N:
        next_arrival = tLL if n_arr < N else math.inf

        # ---- Caso 1: llegada (tLL es el menor) ----
        if next_arrival <= t1 and next_arrival <= t2:
            t = next_arrival
            cid = n_arr
            n_arr += 1
            arr1[cid] = t

            if not server1_busy:
                server1_busy = True
                current1 = cid
                s1_start[cid] = t
                t1 = t + rng.exponential(1.0 / mu1)  # fin de serv1
            else:
                q1.append(cid)

            if n_arr < N:
                tLL = t + rng.exponential(1.0 / lambda_rate)  # siguiente llegada

        # ---- Caso 2: fin de servicio en S1 (t1 es el menor) ----
        elif t1 <= t2:
            t = t1
            cid = current1

            dep1[cid] = t
            arr2[cid] = t  # entra a S2 justo al salir de S1

            # entra a S2
            if not server2_busy:
                server2_busy = True
                current2 = cid
                s2_start[cid] = t
                t2 = t + rng.exponential(1.0 / mu2)  # fin de serv2
            else:
                q2.append(cid)

            # libera/continúa S1
            if q1:
                nxt = q1.popleft()
                current1 = nxt
                s1_start[nxt] = t
                t1 = t + rng.exponential(1.0 / mu1)
                server1_busy = True
            else:
                server1_busy = False
                current1 = None
                t1 = math.inf

        # ---- Caso 3: fin de servicio en S2 (t2 es el menor) ----
        else:
            t = t2
            cid = current2

            dep2[cid] = t
            n_dep += 1

            # libera/continúa S2
            if q2:
                nxt = q2.popleft()
                current2 = nxt
                s2_start[nxt] = t
                t2 = t + rng.exponential(1.0 / mu2)
                server2_busy = True
            else:
                server2_busy = False
                current2 = None
                t2 = math.inf

    # Métricas (descartando warmup)
    start = warmup
    sl = slice(start, N)

    W  = (dep2[sl] - arr1[sl]).mean()
    Wq1 = (s1_start[sl] - arr1[sl]).mean()
    Wq2 = (s2_start[sl] - arr2[sl]).mean()
    Wq = Wq1 + Wq2

    return {
        "W_sim": float(W),
        "Wq_sim": float(Wq),
        "Wq1_sim": float(Wq1),
        "Wq2_sim": float(Wq2),
    }


def analytic_tandem_mm1(lambda_rate, mu1, mu2):
    """
    Resultado analítico (red de Jackson / Burke):
    Cada nodo se comporta como un M/M/1 con llegada λ.
      W_i  = 1/(mu_i - λ)
      Wq_i = W_i - 1/mu_i
    Totales: W = W1+W2, Wq = Wq1+Wq2
    """
    if not (lambda_rate < mu1 and lambda_rate < mu2):
        raise ValueError("Sistema inestable: se requiere lambda < mu1 y lambda < mu2.")

    W1 = 1.0 / (mu1 - lambda_rate)
    W2 = 1.0 / (mu2 - lambda_rate)
    W  = W1 + W2

    Wq1 = W1 - 1.0 / mu1
    Wq2 = W2 - 1.0 / mu2
    Wq  = Wq1 + Wq2

    return {"W": W, "Wq": Wq, "Wq1": Wq1, "Wq2": Wq2}


if __name__ == "__main__":
    # Ejemplo (estable): λ < μ1 y λ < μ2
    lam, mu1, mu2 = 0.8, 1.3, 1.4

    sim = simulate_tandem_mm1(lam, mu1, mu2, N=200_000, warmup=5_000, seed=7)
    ana = analytic_tandem_mm1(lam, mu1, mu2)

    print("Simulacion:", sim)
    print("Analitico:", ana)


Simulacion: {'W_sim': 3.6344925825164953, 'Wq_sim': 2.1550632994565344, 'Wq1_sim': 1.2108570898529392, 'Wq2_sim': 0.9442062096035952}
Analitico: {'W': 3.666666666666667, 'Wq': 2.1831501831501834, 'Wq1': 1.2307692307692308, 'Wq2': 0.9523809523809527}
